# Notebook 2 — Are forecast intervals trustworthy in production?

**Headline question for a dispatch desk:** Can we trust Week 3 P10–P90 bands as an **80% envelope** for regulating reserve?

Five beats:

1. **Method** — split conformal + CQR, chronological calibration
2. **CQR setup** — quantile LightGBM, coverage table, one-week fan plot
3. **Coverage over time** — rolling-origin backtest and money chart
4. **Width trade-off** — month/regime breakdowns and pinball sharpness
5. **Market narrative** — reserve, imbalance, and what would break

See also: [`docs/conformal_mental_model.md`](../docs/conformal_mental_model.md)

In [ ]:
import sys
from pathlib import Path

repo_root = Path.cwd()
if not (repo_root / "src").exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import matplotlib.pyplot as plt
import pandas as pd

from src.conformal.backtest import (
    RollingOriginConfig,
    coverage_by_month,
    coverage_by_regime,
    pinball_comparison,
    rolling_origin_backtest,
)
from src.conformal.cqr import run_conformal_cqr
from src.conformal.quantile_lgbm import WEEK3_MODEL_PARAMS, QuantileLGBM
from src.conformal.simulate import WindSimulationConfig, feature_columns, simulate_wind_forecast
from src.conformal.split import chronological_conformal_split
from src.plotting import (
    COLOR_ACTUAL,
    COLOR_CQR,
    COLOR_NOMINAL,
    COLOR_RAW,
    save_summary_figure,
    setup_style,
)

setup_style()
wind_parquet = (
    repo_root.parent
    / "wind-quantile-forecast"
    / "data"
    / "processed"
    / "day_ahead_wind.parquet"
)


def _load_wind_frame(parquet_path: Path, sim_config: WindSimulationConfig) -> pd.DataFrame:
    """Load sibling Week 3 parquet when available; else synthetic wind."""
    if parquet_path.exists():
        try:
            frame = pd.read_parquet(parquet_path)
            print(f"Loaded Week 3 parquet: {len(frame):,} rows")
            return frame
        except ImportError:
            print("Parquet engine unavailable — using synthetic wind.")
    frame = simulate_wind_forecast(sim_config)
    print(f"Using synthetic wind: {len(frame):,} hourly rows")
    return frame

## 1. Method

Week 3 trains separate LightGBM models at **P10, P50, P90** with pinball loss. Default models often **under-cover** (~59% empirical vs 80% nominal on June 2019 CV).

**Split conformal / CQR:** score nonconformity on a held-out calibration set, take a quantile of those scores, and widen future intervals until **marginal coverage** matches nominal — no distributional assumption on errors.

**Fine print:** guarantees need approximate **exchangeability**. We use a **chronological** train → calibration → test split (24 h gap), not a random shuffle. Rolling re-calibration below tracks slow drift better than one slice.

## 2. CQR setup

Synthetic DE-style day-ahead wind (~120 days) with heteroskedastic tails — the Week 3 under-coverage story in miniature. If a sibling `wind-quantile-forecast/data/processed/day_ahead_wind.parquet` exists locally, swap it in here.

In [ ]:
frame = _load_wind_frame(wind_parquet, WindSimulationConfig(n_days=120, seed=7))

cols = feature_columns()
train, cal, test = chronological_conformal_split(frame, gap_hours=24)
print(f"Train={len(train):,}  Cal={len(cal):,}  Test={len(test):,}")

model = QuantileLGBM(model_params=WEEK3_MODEL_PARAMS)
result = run_conformal_cqr(train, cal, test, cols, model=model)

In [ ]:
rows = [
    {
        "band": "Raw P10–P90",
        "nominal": "80%",
        "empirical": result.raw_80.coverage,
        "gap_pp": result.raw_80.coverage_gap * 100,
        "mean_width_mw": result.raw_80.mean_width,
    },
    {
        "band": "CQR 80%",
        "nominal": "80%",
        "empirical": result.cqr_80.coverage,
        "gap_pp": result.cqr_80.coverage_gap * 100,
        "mean_width_mw": result.cqr_80.mean_width,
    },
    {
        "band": "Raw P05–P95",
        "nominal": "90%",
        "empirical": result.raw_90.coverage,
        "gap_pp": result.raw_90.coverage_gap * 100,
        "mean_width_mw": result.raw_90.mean_width,
    },
    {
        "band": "CQR 90%",
        "nominal": "90%",
        "empirical": result.cqr_90.coverage,
        "gap_pp": result.cqr_90.coverage_gap * 100,
        "mean_width_mw": result.cqr_90.mean_width,
    },
]
coverage_table = pd.DataFrame(rows)
(
    coverage_table.style.format(
        {
            "empirical": "{:.1%}",
            "gap_pp": "{:+.0f}",
            "mean_width_mw": "{:,.0f}",
        }
    )
    .set_caption("Held-out test block: raw quantile vs CQR")
    .hide(axis="index")
)

**Typical finding:** raw quantiles **under-cover** (negative gap); CQR hits nominal (± finite-sample noise) by **widening** intervals.

In [ ]:
plot_df = result.test_frame.sort_values("valid_time").head(168)
t = plot_df["valid_time"]
y = plot_df["wind_mw"]

fig, ax = plt.subplots(figsize=(12, 4.5))
ax.fill_between(t, plot_df["raw_p10"], plot_df["raw_p90"], alpha=0.25, color=COLOR_RAW, label="Raw P10–P90")
ax.fill_between(t, plot_df["cqr80_lo"], plot_df["cqr80_hi"], alpha=0.25, color=COLOR_CQR, label="CQR 80%")
ax.plot(t, y, color=COLOR_ACTUAL, linewidth=0.8, label="Actual")
ax.plot(t, plot_df["pred_p50"], color=COLOR_CQR, linewidth=0.8, label="P50")
ax.set_ylabel("Wind (MW)")
ax.set_title("One week: raw vs conformalized 80% envelope")
ax.legend(loc="upper right")
fig.autofmt_xdate()
fig.tight_layout()
plt.show()

## 3. Coverage over time

Rolling-origin evaluation over the **longest window the data allows**: expanding train, fixed calibration and test blocks (30-day step), 24 h gaps between blocks.

Synthetic wind uses **365 days** with seasonal heteroskedasticity and a mid-sample residual-scale jump so coverage can drift by month — the story a reserve desk needs to see, not just one held-out slice.

In [ ]:
long_frame = _load_wind_frame(
    wind_parquet,
    WindSimulationConfig(
        n_days=365,
        seed=7,
        seasonal_heteroskedasticity=True,
        drift_day=180,
        drift_scale_factor=1.35,
    ),
)
print(f"Long backtest rows: {len(long_frame):,}")

backtest_cfg = RollingOriginConfig()
FAST_PARAMS = {"n_estimators": 80, "num_leaves": 65, "verbosity": -1}

backtest = rolling_origin_backtest(
    long_frame,
    cols,
    config=backtest_cfg,
    model_params=FAST_PARAMS,
)
print(f"Folds: {len(backtest.fold_table)}")
backtest.fold_table[
    [
        "fold_id",
        "test_month",
        "raw_80_coverage",
        "cqr_80_coverage",
        "raw_80_width",
        "cqr_80_width",
        "n_test",
    ]
]

Each point is one rolling test block. A desk monitors this: **does empirical coverage track the nominal line month after month?** Raw quantiles that sag below 80% are unpriced shortage risk.

In [ ]:
fold_plot = backtest.fold_table.sort_values("test_month")

fig, ax = plt.subplots(figsize=(12, 4.5))
ax.plot(
    fold_plot["test_month"],
    fold_plot["raw_80_coverage"],
    marker="o",
    label="Raw P10–P90",
    color=COLOR_RAW,
)
ax.plot(
    fold_plot["test_month"],
    fold_plot["cqr_80_coverage"],
    marker="s",
    label="CQR 80%",
    color=COLOR_CQR,
)
ax.axhline(0.80, color=COLOR_NOMINAL, linestyle="--", linewidth=1, label="Nominal 80%")
ax.set_ylabel("Empirical coverage")
ax.set_xlabel("Test block start month")
ax.set_title("Rolling-origin coverage over time")
ax.set_ylim(0.45, 1.02)
ax.legend(loc="lower left")
plt.xticks(rotation=45, ha="right")
fig.tight_layout()
save_summary_figure(fig, "nb02_coverage_over_time")
plt.close(fig)

## 4. Width trade-off

CQR buys **coverage** with **width** — more MW held around P50. Aggregate over all rolling test hours. **Regime** = tercile of NWP hub wind speed (low / mid / high) — the hours where reserve sizing hurts most.

Pinball loss confirms CQR widens for coverage, not to improve median sharpness (P50 unchanged; tail pinball often worse — expected).

In [ ]:
width_summary = backtest.fold_table.assign(
    width_gain_mw=lambda d: d["cqr_80_width"] - d["raw_80_width"],
    coverage_gain_pp=lambda d: (d["cqr_80_coverage"] - d["raw_80_coverage"]) * 100,
)[
    ["test_month", "raw_80_width", "cqr_80_width", "width_gain_mw", "raw_80_coverage", "cqr_80_coverage", "coverage_gain_pp"]
]
(
    width_summary.style.format(
        {
            "raw_80_width": "{:,.0f}",
            "cqr_80_width": "{:,.0f}",
            "width_gain_mw": "{:+,.0f}",
            "raw_80_coverage": "{:.1%}",
            "cqr_80_coverage": "{:.1%}",
            "coverage_gain_pp": "{:+.0f}",
        }
    )
    .set_caption("Fold-level width vs coverage trade-off (80% band)")
    .hide(axis="index")
)

In [ ]:
month_table = coverage_by_month(backtest.predictions).round(3)
regime_table = coverage_by_regime(backtest.predictions).round(3)
print("Coverage by calendar month:")
month_table

In [ ]:
print("Coverage by wind regime:")
regime_table

In [ ]:
pinball_table = pinball_comparison(backtest.predictions)
pinball_table.round(1)

## 5. Market narrative

**Guaranteed-coverage intervals are what a trading desk or reserve-planning team can actually contract against.**

A model that claims **80%** but delivers **~65%** creates real financial exposure:

| Risk | Mechanism |
|---|---|
| **Under-procured reserve** | P10–P90 sized as an 80% envelope but covering only ~65% of hours → more redispatch and balancing activation |
| **Unpriced imbalance** | Actual wind outside the band more often than the label implies → settlement exposure on the uncovered tail |
| **False confidence in limits** | Traders or optimizers treat the band as a hard constraint; under-coverage breaks that constraint silently |

**CQR** pays in **width** (more MW held around P50) and buys a label you can write into a reserve or imbalance policy. It does not promise the narrowest band — it promises the **stated coverage level** holds on the calibration window (and rolling re-calibration tracks slow drift better than one split).

Illustrative scaling (not a full cost model):

- Reserve slice ≈ `(P90 − P10) / 2` MW per hour (half the 80% band)
- If raw bands under-cover by **15 percentage points**, roughly **15% more hours** fall outside the band than planned
- At **500 MW** mean reserve and **€120/MWh** imbalance premium for those surprise hours → **€9,000/hour** of unplanned exposure per 100 MW of reserve error (order-of-magnitude illustration)

**Decision read:** monitor the **coverage-over-time chart** the way Notebook 1 monitors placebo p-values. If raw coverage sags in winter or high-wind regimes, do **not** size regulating reserve from the raw quantile label alone — conformalize or hold extra flex explicitly.

**What would break:** exchangeability under drift (rolling cal is a patch, not a proof); marginal 80% ≠ conditional 80% in every high-wind hour; CQR does not fix stale NWP or broken lags; re-run on real Week 3 parquet for production claims. For Bayesian credible bands on short solar horizons, see Notebook 3 — credible ≠ coverage-guaranteed.